In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque
import gym
import requests
import matplotlib.pyplot as plt
from gym_duckietown.envs import DuckietownEnv
from dqn import DQN
from agent import Agent
from wrapper import extract_state, PerceptionModule

In [19]:
training_maps = [
    "4way.yaml",
    "loop_dyn_duckiebots.yaml",
    "loop_empty.yaml",
    "loop_obstacles.yaml",
    "loop_only_duckies.yaml",
    "regress_4way_adam.yaml",
    "regress_4way_drivable.yaml",
    "small_loop_cw.yaml",
    "small_loop_only_duckies.yaml",
    "small_loop.yaml",
    "straight_road.yaml",
    "udem1.yaml",
    "zigzag_dists.yaml"
]

In [20]:
for map_name in training_maps:
    env = gym.make("Duckietown-udem1-v0", map_name=map_name, camera_width=640, camera_height=640)

INFO:gym-duckietown:Information about the graphics card:
 pyglet_version: 1.5.27
    information: dict[4]
                 │ vendor: NVIDIA Corporation
                 │ renderer: NVIDIA GeForce RTX 4050 Laptop GPU/PCIe/SSE2
                 │ version: 4.6.0 NVIDIA 535.183.01
                 │ shading-language-version: 4.60 NVIDIA
  nvidia_around: True


DEBUG:gym-duckietown:loading map file "4way.yaml"
INFO:gym-duckietown:done
DEBUG:gym-duckietown:[    0.62208           0     0.82677] corresponds to tile at (1, 1) which is not drivable: {'coords': (1, 1), 'kind': 'asphalt', 'angle': 1, 'drivable': False, 'texture': <gym_duckietown.graphics.Texture object at 0x7f59cb0ef0a0>, 'color': array([    0.81619,      1.1356,     0.89211,           1])}
DEBUG:gym-duckietown:Invalid pose. Collision free: True On drivable area: False
DEBUG:gym-duckietown:safety_factor: 1.3
DEBUG:gym-duckietown:pos: [    0.53738           0      0.7785]
DEBUG:gym-duckietown:l_pos: [    0.62208           0     0.82677]
DEBUG:gym-duckietown:r_pos: [    0.45267           0     0.73022]
DEBUG:gym-duckietown:f_pos: [    0.47944           0     0.88015]
DEBUG:gym-duckietown:No tile found at [ -0.0086006           0     0.81381] (-1, 1)
DEBUG:gym-duckietown:Invalid pose. Collision free: True On drivable area: False
DEBUG:gym-duckietown:safety_factor: 1.3
DEBUG:gym-duckiet

In [6]:
env = gym.make("Duckietown-udem1-v0", camera_width=640, camera_height=640)
obs = env.reset()
agent = Agent(input_dim=4, action_dim=3)  # vezi `extract_state`
perception_module = PerceptionModule("slow_color.pt", "fine_tune_myset.pth")

INFO:gym-duckietown:Information about the graphics card:
 pyglet_version: 1.5.27
    information: dict[4]
                 │ vendor: NVIDIA Corporation
                 │ renderer: NVIDIA GeForce RTX 4050 Laptop GPU/PCIe/SSE2
                 │ version: 4.6.0 NVIDIA 535.183.01
                 │ shading-language-version: 4.60 NVIDIA
  nvidia_around: True


DEBUG:gym-duckietown:loading map file "udem1.yaml"
INFO:gym-duckietown:done
INFO:gym-duckietown:Starting at [     3.8777           0     0.99105] 4.549442904792602
INFO:gym-duckietown:using DuckietownEnv
/home/plapusk/Documents/Licenta/Trainer_labels/gym-duckietown/duckie-env-py38/lib/python3.8/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/plapusk/Documents/Licenta/Trainer_labels/gym-duckietown/duckie-env-py38/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/home/plapusk/Documents/Licenta/Trainer_labels/gym-duckietown/duckie-env-py38/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:187: UserWarning: WARN: Future gym versions will req

In [ ]:
MAX_STEPS_PER_EPISODE = 500 

for episode in range(10):
    obs = env.reset()
    done = False
    total_reward = 0
    step = 0

    while not done and step < MAX_STEPS_PER_EPISODE:
        step += 1
        # Trimite imaginea la serverul Flask

        bboxes, seg_mask, _ = perception_module.predict(obs)

        state = extract_state(bboxes, seg_mask)

        action_id = agent.select_action(state)
        if action_id == 0:
            action = [0.0, -1.0]  # stânga
        elif action_id == 1:
            action = [0.44, 0.0]  # înainte
        else:
            action = [0.0, 1.0]  # dreapta

        obs, reward, done, _ = env.step(action)
        total_reward += reward

        # Obține next_state
        next_bboxes, next_seg_mask, _ = perception_module.predict(obs)
        next_state = extract_state(next_bboxes, next_seg_mask)

        agent.store_transition(state, action_id, reward, next_state, done)
        agent.train()
        agent.decay_epsilon()

    agent.update_target()
    print(f"Ep {episode}, reward: {total_reward:.2f}")

DEBUG:gym-duckietown:[     3.5859           0     0.58359] corresponds to tile at (6, 0) which is not drivable: {'coords': (6, 0), 'kind': 'floor', 'angle': 1, 'drivable': False, 'texture': <gym_duckietown.graphics.Texture object at 0x7f59b3dd7af0>, 'color': array([    0.82186,      0.9027,     0.94325,           1])}
DEBUG:gym-duckietown:Invalid pose. Collision free: True On drivable area: False
DEBUG:gym-duckietown:safety_factor: 1.3
DEBUG:gym-duckietown:pos: [     3.6832           0     0.59002]
DEBUG:gym-duckietown:l_pos: [     3.7804           0     0.59644]
DEBUG:gym-duckietown:r_pos: [     3.5859           0     0.58359]
DEBUG:gym-duckietown:f_pos: [     3.6754           0     0.70676]
INFO:gym-duckietown:Starting at [     4.0145           0     0.69893] 2.9756732092858327


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

In [29]:
torch.save(agent.model.state_dict(), 'agent_model.pth')

In [16]:
print([env_id for env_id in gym.envs.registry.keys() if "Duckietown" in env_id])

['Duckietown-MOOC_modcon-v0', 'Duckietown-ETHZ_autolab_fast_track-v0', 'Duckietown-ETHZ_loop-v0', 'Duckietown-zigzag_dists_bordered-v0', 'Duckietown-TTIC_ripltown-v0', 'Duckietown-ETHZ_autolab_technical_track_bordered-v0', 'Duckietown-ETH_small_loop_3-v0', 'Duckietown-ETH_small_intersect-v0', 'Duckietown-robotarium2-v0', 'Duckietown-straight_road-v0', 'Duckietown-calibration_map_ext-v0', 'Duckietown-4way_bordered-v0', 'Duckietown-ETH_small_loop_1-v0', 'Duckietown-ETH_intersection_map-v0', 'Duckietown-ETU_autolab_track-v0', 'Duckietown-straight_road_down-v0', 'Duckietown-ETH_small_loop_2_bordered-v0', 'Duckietown-ETHZ_loop_bordered-v0', 'Duckietown-calibration_map_int-v0', 'Duckietown-small_loop-v0', 'Duckietown-TTIC_loop-v0', 'Duckietown-loop_obstacles-v0', 'Duckietown-ETH_large_intersect-v0', 'Duckietown-loop_dyn_duckiebots-v0', 'Duckietown-udem1_empty-v0', 'Duckietown-small_loop_cw-v0', 'Duckietown-experiment_loop-v0', 'Duckietown-ETHZ_autolab_technical_track-v0', 'Duckietown-udem1-v